# Quantitatively Evaluating Diffusion Models

Evaluation of generative models like [Stable Diffusion](https://huggingface.co/docs/diffusers/stable_diffusion) is subjective in nature.

Here is an example of using a CLIP-based quantitative approach to evaluate Diffusion models.

Adapted from:
https://huggingface.co/docs/diffusers/en/conceptual/evaluation

Updated: Febuary 2026

In [1]:
%%capture
!pip install -q datasets diffusers transformers accelerate torchmetrics[image]
!pip install --upgrade torchmetrics
!pip install --upgrade transformers

### Text-guided image generation

[CLIP score](https://arxiv.org/abs/2104.08718) measures the compatibility of image-caption pairs. Higher CLIP scores imply higher compatibility 🔼. The CLIP score is a quantitative measurement of the qualitative concept "compatibility". Image-caption pair compatibility can also be thought of as the semantic similarity between the image and the caption. CLIP score was found to have high correlation with human judgement.

Generate some images with multiple prompts:

In [2]:
from diffusers import StableDiffusionPipeline
import torch

model_ckpt = "CompVis/stable-diffusion-v1-4"
sd_pipeline = StableDiffusionPipeline.from_pretrained(model_ckpt, torch_dtype=torch.float16).to("cuda")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--CompVis--stable-diffusion-v1-4/snapshots/133a221b8aa7292a167afc5127cb63fb5005638b/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--CompVis--stable-diffusion-v1-4/snapshots/133a221b8aa7292a167afc5127cb63fb5005638b/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
prompts = [
    "a photo of an astronaut riding a horse on mars",
    "A high tech solarpunk utopia in the Amazon rainforest",
    "A pikachu fine dining with a view to the Eiffel Tower",
    "A mecha robot in a favela in expressionist style",
    "an insect robot preparing a delicious meal",
    "A small cabin on top of a snowy mountain in the style of Disney, artstation",
]

images = sd_pipeline(prompts, num_images_per_prompt=1, output_type="numpy").images
print(images.shape)

  0%|          | 0/50 [00:00<?, ?it/s]

(6, 512, 512, 3)


/usr/local/lib/python3.12/dist-packages/diffusers/image_processor.py:775: FutureWarning: the output_type numpy is outdated and has been set to `np`. Please make sure to set it to one of these instead: `pil`, `np`, `pt`, `latent`
  deprecate("Unsupported output_type", "1.0.0", deprecation_message, standard_warn=False)


First, a few important fuction definitions to calcualte the CLIP scores.

In [5]:
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import torch
import numpy as np

def get_clip_model_and_processor():
    global _clip_model, _clip_processor
    if _clip_model is None or _clip_processor is None:
        print(f"Loading CLIP model '{_clip_model_name}' to {device}...")
        _clip_model = CLIPModel.from_pretrained(_clip_model_name).to(device)
        _clip_processor = CLIPProcessor.from_pretrained(_clip_model_name)
    return _clip_model, _clip_processor



def calculate_clip_score(images, prompts):
    model, processor = get_clip_model_and_processor()

    # Convert numpy images (0-1 float) to PIL images (0-255 uint8)
    # 'images' is a numpy array of shape (batch_size, H, W, 3) with values in [0, 1]
    images_pil = [Image.fromarray((img_arr * 255).astype(np.uint8)) for img_arr in images]

    # Process images and texts
    inputs = processor(
        text=prompts,
        images=images_pil,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        # Call the model's forward method directly and access the embeds
        # CLIPOutput contains image_embeds and text_embeds
        outputs = model(**inputs, return_dict=True)
        image_features = outputs.image_embeds
        text_features = outputs.text_embeds

    image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True) # Normalize
    text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True) # Normalize

    # Calculate cosine similarity (dot product of normalized vectors)
    similarity_score = (image_features * text_features).sum(dim=-1)

    # Return the mean similarity score, rounded to 4 decimal places
    return round(float(similarity_score.mean().detach().cpu()), 4)

Now, perform the CLIP calculation

In [6]:
# Define device explicitly for this cell, assuming CUDA is preferred if available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load the CLIP model and processor once
_clip_model = None
_clip_processor = None
_clip_model_name = "openai/clip-vit-base-patch16" # Match the original model name

# Calculate_clip_score
sd_clip_score = calculate_clip_score(images, prompts)
print(f"CLIP score: {sd_clip_score}")

Using device: cuda
Loading CLIP model 'openai/clip-vit-base-patch16' to cuda...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch16
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


CLIP score: 0.35


In the above example, we generated one image per prompt. If we generated multiple images per prompt, we would have to take the average score from the generated images per prompt.

If we wanted to compare two checkpoints compatible with the [`StableDiffusionPipeline`](https://huggingface.co/docs/diffusers/api/pipelines/stable_diffusion/overview) we should pass a generator while calling the pipeline. First, we generate images with a fixed seed with the [v1-4 Stable Diffusion checkpoint](https://huggingface.co/CompVis/stable-diffusion-v1-4):


In [7]:
seed = 0
generator = torch.manual_seed(seed)
images = sd_pipeline(prompts, num_images_per_prompt=1, generator=generator, output_type="numpy").images

  0%|          | 0/50 [00:00<?, ?it/s]

Then we load the [v1-5 checkpoint](https://huggingface.co/runwayml/stable-diffusion-v1-5) to generate images:

In [8]:
model_ckpt_1_5 = "runwayml/stable-diffusion-v1-5"
weight_dtype = torch.float16 # Define weight_dtype here
sd_pipeline_1_5 = StableDiffusionPipeline.from_pretrained(model_ckpt_1_5, torch_dtype=weight_dtype).to(device)
images_1_5 = sd_pipeline_1_5(prompts, num_images_per_prompt=1, generator=generator, output_type="numpy").images

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  0%|          | 0/50 [00:00<?, ?it/s]

And finally, we compare their CLIP scores:

In [10]:
sd_clip_score_1_4 = calculate_clip_score(images, prompts)
print(f"CLIP Score with version-1-4: {sd_clip_score_1_4}")

sd_clip_score_1_5 = calculate_clip_score(images_1_5, prompts)
print(f"CLIP Score with version-1-5: {sd_clip_score_1_5}")

CLIP Score with version-1-4: 0.3486
CLIP Score with version-1-5: 0.3562


It seems like the [version 1-5](https://huggingface.co/runwayml/stable-diffusion-v1-5) checkpoint performs better than version 1-4. The number of prompts we used to compute the CLIP scores is quite low. For a more practical evaluation, this number should be way higher, and the prompts should be more diverse.
